### 1. load datasets

In [5]:
import pandas as pd

import pandas as pd
df_mr = pd.read_csv('../datasets/translation/pure_marathi.csv')
print(df_mr.shape)
df_mr.head()

(64, 5)


,scheme_id,site,scheme_name,description,scheme_link
0,1,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,आनंदाचा शिधा,दि. 04.10.2022 च्या शासन निर्णयानुसार राष्ट्री...,https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4...
1,2,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,एपीएल शेतकरी,"राज्यातील छत्रपती संभाजीनगर, जालना, नांदेड, बी...",https://mahafood.gov.in/scheme/%e0%a4%8f%e0%a4...
2,3,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,शिवभोजन,राज्यातील गरीब व गरजू जनतेला सवलतीच्या दरात भो...,https://mahafood.gov.in/scheme/%e0%a4%b6%e0%a4...
3,4,https://maharashtra.gov.in/Site/1604/scheme,कृषी योजना,शेतकरी वर्गासाठी राज्य शासनातर्फे अनेक योजना उ...,https://www.manage.gov.in/fpoacademy/SGSchemes...
4,5,https://maharashtra.gov.in/Site/1604/scheme,कृषी तारण कर्ज योजना,शेतकऱ्याला असलेल्या आर्थिक गरजेपोटी तसेच स्थान...,https://www.msamb.com/Schemes/PledgeFinance


### 2. connect elasticsearch

In [12]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())

{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


  ### 3. load models

In [15]:
import sys, os
sys.path.append(os.path.abspath("../loadModels"))

In [1]:
from loadModels import load_mahaSBERT

mahaSBERT_model = load_mahaSBERT()
print("mahaSBERT model loaded successfully")


mahaSBERT model loaded successfully


In [2]:
from loadModels import load_indicSBERT

indicSBERT_model = load_indicSBERT()

print("indicSBERT model loaded successfully")

indicSBERT model loaded successfully


### 4. generate embeddings


In [6]:
df_mr["mahasbert_des_vector"] = df_mr["description"].apply(lambda x: mahaSBERT_model.encode(x))

In [7]:
df_mr["indicsbert_des_vector"] = df_mr["description"].apply(lambda x: indicSBERT_model.encode(x))

### 5. generate mappings

In [28]:
from indexMappings import indexMappings

es.indices.create(index = "marathi" , mappings = indexMappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'marathi'})

### 6. Generate records

In [8]:
marathi_record_list = df_mr.to_dict("records")

In [9]:
marathi_record_list[0]["mahasbert_des_vector"]

array([-3.43049914e-02, -5.12936059e-03,  2.20858175e-02,  7.74219539e-03,
        2.82565653e-02, -9.91985109e-03, -4.03571315e-03,  9.23226029e-03,
        8.11281335e-03,  1.03322195e-03,  8.72045173e-04,  1.06196264e-02,
        9.83992498e-03, -1.81763042e-02, -5.25347982e-03, -2.00595502e-02,
       -5.52726770e-03, -7.68145919e-03,  3.83548182e-03, -1.97888520e-02,
        6.15616888e-03, -1.16232196e-02, -1.32690128e-02,  1.02178743e-02,
        9.95635428e-03,  1.41091328e-02,  3.36631276e-02, -1.12010678e-02,
        1.58473086e-02,  1.24502508e-02, -1.64590012e-02, -1.52322641e-02,
        1.11759501e-02, -1.24793975e-02, -1.47179316e-03, -2.52346764e-03,
       -2.11245753e-03, -1.46145839e-02,  1.93673894e-02, -1.32602537e-02,
        1.07882898e-02, -8.99164937e-03, -1.66752767e-02, -1.32896462e-02,
       -4.18951269e-04, -2.33620219e-02,  1.63732078e-02,  1.80353094e-02,
       -7.73714250e-03, -3.30218533e-03, -1.03246048e-02, -1.90420318e-02,
       -1.08179897e-02, -

In [10]:
marathi_record_list[0]["indicsbert_des_vector"]

array([-2.75625903e-02, -4.21745889e-03,  1.04450770e-02, -4.55878768e-03,
        1.42357191e-02, -1.03320845e-03,  9.56011750e-03,  4.62173810e-03,
        1.35150820e-03, -3.93428188e-03,  1.17953606e-02,  6.31843694e-03,
        9.00148600e-03, -7.88422010e-04, -4.03874740e-03, -9.08759795e-03,
       -5.82811749e-03, -4.74030618e-03, -1.24796070e-02, -6.68922300e-03,
        6.53981930e-03,  3.59811704e-03, -3.02671418e-02, -8.72205070e-04,
        3.87123146e-04,  3.81355733e-03,  1.62489451e-02, -1.29870297e-02,
        2.62304937e-04,  4.41640103e-03, -1.36055974e-02, -1.26155205e-02,
        5.96339023e-03,  1.97773473e-03,  7.50570837e-03, -7.19394069e-03,
       -1.35319931e-02,  5.44150220e-03,  6.76344521e-03,  3.76271373e-06,
        4.60880157e-03, -1.81529690e-02, -3.08287218e-02, -1.56393480e-02,
       -7.46813929e-03, -1.58082265e-02,  1.72250755e-02,  2.11984757e-02,
        1.40023781e-02,  1.00450416e-03, -2.59756614e-02, -1.10672368e-02,
        5.44762518e-03, -

In [35]:
for record in marathi_record_list:
    try:
        es.index(index="marathi", document=record, id=record['scheme_id'])
    except Exception as e:
        print("error", e)

In [13]:
es.count(index="marathi")

ObjectApiResponse({'count': 64, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})